# Webcam Discovery Colab Validation (Location-Agnostic)
Set `QUERY` and optionally `SAMPLE_STREAMS`. This notebook runs real discovery and audits artifacts.

In [ ]:
QUERY = "public live webcams near Eiffel Tower"
SAMPLE_STREAMS = []
BRANCH = "main"
REPO_URL = "https://github.com/dshipley71/webcam-discovery.git"


In [ ]:
import os
if not os.path.exists('webcam-discovery'):
    get_ipython().system('git clone {REPO_URL}')
get_ipython().run_line_magic('cd', 'webcam-discovery')
get_ipython().system('git fetch --all')
get_ipython().system('git checkout {BRANCH}')
get_ipython().system('git pull --ff-only')
get_ipython().system('python -m pip install -U pip')
get_ipython().system('python -m pip install -e .')


In [ ]:
import os
from google.colab import userdata
os.environ['WCD_OLLAMA_API_KEY'] = userdata.get('WCD_OLLAMA_API_KEY')
os.environ['WCD_OLLAMA_BASE_URL'] = userdata.get('WCD_OLLAMA_BASE_URL') or 'https://ollama.com'
print('Secrets loaded:', bool(os.environ.get('WCD_OLLAMA_API_KEY')))


In [ ]:
import subprocess, threading, time
cmd=['webcam-discovery','run-agentic',QUERY,'--enable-feed-discovery','--max-feed-endpoints','100','--max-feed-records','3000','--max-stream-candidates','2500','--per-source-stream-cap','500','--preserve-direct-streams','--max-streams','500']
print('Running:', ' '.join(cmd))
p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
last=[time.time()]
def reader():
    for line in p.stdout:
        print(line.rstrip())
        last[0]=time.time()
threading.Thread(target=reader,daemon=True).start()
while p.poll() is None:
    time.sleep(5)
    if time.time()-last[0] > 30:
        print('[heartbeat] process still running...')
print('Exit:', p.returncode)
if p.returncode != 0:
    raise RuntimeError('run-agentic failed')
